In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [2]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [3]:
from openai import OpenAI

openai_client = OpenAI()

In [4]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [22]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I still enroll FAQ"}', call_id='call_AUnurEW0s9wcbeHwixcXLivQ', name='search', type='function_call', id='fc_01088da7c821c857006ac5c859f53887d2bc5f45880e5177a8', caller=None, namespace=None, status='completed')]

In [6]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [7]:
import json

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [23]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    print(f"item: {item}")
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item) #peforms the search through Python
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

item: ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join enrollment late registration"}', call_id='call_pLbQ9bKnpU2eTMUVRdZ76TtA', name='search', type='function_call', id='fc_020d053b8725ceb6006ac5c9dcf73c87d2a2aa06721d37b11d', caller=None, namespace=None, status='completed')
function_call: search {"query":"join course discovered can I join enrollment late registration"}


In [9]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:

    messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model="gpt-5.4-mini",
            input=messages,
            tools=[search_tool],
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                # make_call is to make call to search
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    return last_answer

In [10]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...


function_call: search {"query":"Olama run locally Ollama local install run locally"}
function_call: search {"query":"run Ollama locally course FAQ"}
function_call: search {"query":"local Ollama setup install start server"}
iteration #2...
ASSISTANT:
To run Ollama locally for the course:

1. Install Ollama from **https://ollama.com/download**
   - **macOS**: download the `.pkg`
   - **Windows**: download the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Start a model locally:
   ```bash
   ollama run llama3
   ```
   This downloads the model, starts it locally, and opens a chat interface.

3. Check that the local server is running:
   ```bash
   curl http://localhost:11434
   ```
   You should get a response like:
   ```json
   {"models": [...]}
   ```

4. If you want to use it from Python, install the client:
   ```bash
   pip install ollama
   ```

5. Minimal Python example:
   ```python
   import ollama

   response = ollama.chat(

'To run Ollama locally for the course:\n\n1. Install Ollama from **https://ollama.com/download**\n   - **macOS**: download the `.pkg`\n   - **Windows**: download the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Start a model locally:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model, starts it locally, and opens a chat interface.\n\n3. Check that the local server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response like:\n   ```json\n   {"models": [...]}\n   ```\n\n4. If you want to use it from Python, install the client:\n   ```bash\n   pip install ollama\n   ```\n\n5. Minimal Python example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": your_prompt}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection issue during the homework, you can rest

In [11]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered can I still join enrolment registration late join"}
function_call: search {"query":"can I still join the course late FAQ discovered course join now"}
function_call: search {"query":"course registration late enrolment still join after start"}
iteration #2...
ASSISTANT:
Yes — you can still join the course. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.

If you want, I can also help with other course logistics.


'Yes — you can still join the course. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.\n\nIf you want, I can also help with other course logistics.'

In [12]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit course FAQ"}
iteration #2...
function_call: search {"query":"queen gambit chess FAQ queen's gambit"}
iteration #3...
ASSISTANT:
I couldn’t find anything in the course FAQ about “queen gambit,” so it’s likely off-topic or not a course term.

If you meant **Queen’s Gambit** in chess, I’d need to stay within the course FAQ and can’t answer that from the available material.

Is there another course-related area you want to explore?


'I couldn’t find anything in the course FAQ about “queen gambit,” so it’s likely off-topic or not a course term.\n\nIf you meant **Queen’s Gambit** in chess, I’d need to stay within the course FAQ and can’t answer that from the available material.\n\nIs there another course-related area you want to explore?'

In [13]:
# Starting lesson 1.15

agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [14]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [15]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [16]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [17]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [18]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

type(result)

-> Response received


-> Response received


toyaikit.chat.runners.LoopResult

In [19]:
# continue the conversation
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages = result.all_messages,
    callback=callback,
)

-> Response received


-> Response received
